# CPE 342 Term Project — Task 1: Anti-Cheat Pre-Filter (Binary Classification)
**Course:** CPE 342 Machine Learning (Semester 1/2026) — KMUTT  
**System Component:** Karena Thailand Player Intelligence System — Game Integrity Module  
**Task Category:** Supervised Binary Classification (`is_cheater`: `0` = Legitimate, `1` = Cheater)  
**Primary Evaluation Metric:** **$F_2$ Score** ($F_2 = \frac{5 \cdot P \cdot R}{4P + R}$, weighting Recall $2\times$ higher than Precision)  
**Competition Rule Compliance:**
* **Data Usage Policy (Rule 2):** Uses **exclusively** the provided `task1/data/train.csv` for training/validation. `test.csv` is strictly isolated for final inference only (**zero test-set pseudo-labeling or fine-tuning**). Combining Train and Validation sets for final model refitting after validation is explicitly supported.
* **Model Development (Rule 3):** Trained **100% from scratch** on provided tabular data (Logistic Regression, LightGBM, XGBoost, CatBoost). **Zero AutoML frameworks** (`AutoGluon`/`PyCaret` prohibited); hyperparameter search via `Optuna` is supported.
* **Portable Execution:** Runs seamlessly on **Google Colab GPU**, **Kaggle Kernels**, and **Local Workspace (`uv`)**.

## 1. Environment Setup, Hardware Acceleration & Path Resolution
เซลล์นี้ตรวจสอบสภาพแวดล้อมอัตโนมัติ (Local Workspace / Kaggle / Google Colab), ตั้งค่า Random Seed (`SEED = 42`), ตรวจสอบ GPU (`cuda` / `mps` / `cpu`) และกำหนดพาธโฟลเดอร์มาตรฐานตามกฎของ `ml00x` (`data/`, `models/`, `predictions/`, `assets/`)

In [ ]:
import os
import sys
import warnings
from pathlib import Path
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Auto-install optional competitive ML packages on Colab/Kaggle if missing
for pkg in ["lightgbm", "xgboost", "catboost", "optuna"]:
    try:
        __import__(pkg)
    except ImportError:
        if "google.colab" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
            os.system(f"pip install -q {pkg}")

import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    fbeta_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve
)
from scipy.optimize import minimize_scalar

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Multi-environment path resolver (Local / Kaggle / Colab)
BASE_DIR = Path.cwd()
if (BASE_DIR / "project" / "task1").exists():
    TASK_DIR = BASE_DIR / "project" / "task1"
elif (BASE_DIR / "task1").exists():
    TASK_DIR = BASE_DIR / "task1"
else:
    TASK_DIR = BASE_DIR

DATA_DIR = TASK_DIR / "data"
MODELS_DIR = TASK_DIR / "models"
PREDICTIONS_DIR = TASK_DIR / "predictions"
ASSETS_DIR = TASK_DIR / "assets"

for d in [DATA_DIR, MODELS_DIR, PREDICTIONS_DIR, ASSETS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.sans-serif"] = ["Sarabun", "TH Sarabun New", "DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False

print(f"Task Directory : {TASK_DIR}")
print(f"Data Directory : {DATA_DIR}")
print(f"Train Exists   : {(DATA_DIR / 'train.csv').exists()}")
print(f"Test Exists    : {(DATA_DIR / 'test.csv').exists()}")

## 2. Data Loading, Schema Inspection & Missing Target Analysis
จากการตรวจสอบชุดข้อมูลจริงใน `task1/data/train.csv` ($104,000$ แถว $\times 33$ คอลัมน์):
* คอลัมน์ประกอบด้วย `player_id`, ฟีเจอร์เชิงตัวเลข 31 ตัว (เช่น `kill_death_ratio`, `headshot_percentage`, `reaction_time_ms`, `aiming_smoothness`, `spray_control_score`) และตัวแปรเป้าหมาย `is_cheater`
* **ข้อค้นพบสำคัญ (Critical Data Finding):** ใน `train.csv` มีแถวที่ `is_cheater` เป็นค่าว่าง (`NaN`) จำนวน **13,184 แถว** และมีแถวที่มีเฉลยสมบูรณ์จำนวน **90,816 แถว** (`0.0`: 59,091 แถว [65.07%], `1.0`: 31,725 แถว [34.93%]) ดังนั้นต้องคัดกรองเฉพาะแถวที่มี Label สมบูรณ์สำหรับฝึกสอนโมเดล Supervised Learning

In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print(f"Raw Train Shape : {train_raw.shape}")
print(f"Test Shape      : {test_df.shape}")
print("\nTarget (is_cheater) Distribution in Raw Train:")
print(train_raw["is_cheater"].value_counts(dropna=False))

# Separate labeled training rows from unlabeled rows
unlabeled_train_df = train_raw[train_raw["is_cheater"].isna()].copy()
train_df = train_raw[train_raw["is_cheater"].notna()].copy().reset_index(drop=True)
train_df["is_cheater"] = train_df["is_cheater"].astype(int)

ID_COL = "player_id"
TARGET_COL = "is_cheater"
RAW_FEATURES = [c for c in test_df.columns if c != ID_COL]

print(f"\nClean Labeled Train Shape : {train_df.shape} (Positive Ratio: {train_df[TARGET_COL].mean():.4f})")
print(f"Number of Base Features   : {len(RAW_FEATURES)}")

## 3. Exploratory Data Analysis (EDA) & Report Asset Generation
สร้างกราฟวิเคราะห์การกระจายตัวของคลาส (Class Balance), สัดส่วนค่าว่าง (Missing Value Profile) และความสัมพันธ์ของฟีเจอร์หลักต่อการตรวจจับโปรแกรมโกง (Aimbot / Wallhack) พร้อมบันทึกเป็นภาพความละเอียด `300 dpi` ลงใน `task1/assets/`

In [ ]:
# 1. Class distribution & missing value profile plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = train_df[TARGET_COL].value_counts().sort_index()
bars = axes[0].bar(["Legitimate (0)", "Cheater (1)"], counts.values, color=["#2563EB", "#DC2626"], width=0.55)
axes[0].set_title("Task 1: Target Class Distribution (Labeled Train N = 90,816)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Number of Players")
axes[0].grid(axis="y", alpha=0.3)
for bar in bars:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 800, f"{h:,}\n({h/len(train_df)*100:.1f}%)", ha="center", fontweight="bold")

missing_pct = (train_df[RAW_FEATURES].isna().mean() * 100).sort_values(ascending=False).head(12)
axes[1].barh(missing_pct.index[::-1], missing_pct.values[::-1], color="#4F46E5")
axes[1].set_title("Top 12 Features by Missing Value Percentage (%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Missing Percentage (%)")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(ASSETS_DIR / "class-and-missing-distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# 2. Save feature summary table for the technical report
eda_summary = pd.DataFrame({
    "feature": RAW_FEATURES,
    "train_missing_pct": (train_df[RAW_FEATURES].isna().mean() * 100).values,
    "test_missing_pct": (test_df[RAW_FEATURES].isna().mean() * 100).values,
    "legit_mean": train_df[train_df[TARGET_COL] == 0][RAW_FEATURES].mean().values,
    "cheater_mean": train_df[train_df[TARGET_COL] == 1][RAW_FEATURES].mean().values,
})
eda_summary["abs_mean_diff_pct"] = (
    (eda_summary["cheater_mean"] - eda_summary["legit_mean"]).abs()
    / (eda_summary["legit_mean"].abs() + 1e-6) * 100
)
eda_summary = eda_summary.sort_values("abs_mean_diff_pct", ascending=False).reset_index(drop=True)
eda_summary.to_csv(ASSETS_DIR / "eda-feature-summary.csv", index=False)
display(eda_summary.head(10))

## 4. Domain-Specific Anti-Cheat Feature Engineering
ตามคำแนะนำใน `datacard.txt` และพฤติกรรมของโปรแกรมโกง (**Aimbot** และ **Wallhack**):
1. **Aimbot Anomaly Index:** ผู้ใช้ Aimbot มักมี `headshot_percentage` และ `accuracy_score` สูงผิดปกติเมื่อเทียบกับ `reaction_time_ms` หรือมี `aiming_smoothness` ที่แข็งทื่อไม่สอดคล้องกับความแม่นยำ
2. **Wallhack / Pre-Aim Index:** ผู้ใช้ Wallhack จะรู้ตำแหน่งศัตรูล่วงหน้า ทำให้ `first_blood_rate` และ `crosshair_placement` สูงเกินกว่า `map_knowledge` หรือ `account_age_days`
3. **Smurf / Burner Account Velocity:** บัญชีปั๊มหรือบอทจะมี `level_progression_speed` และ `reports_received` ต่ออายุบัญชี (`account_age_days`) สูงมาก รวมถึงไม่ค่อยมีเพื่อนในเกม (`friend_network_size`)
4. **Missingness Indicators:** จำนวนค่าว่างต่อแถว (`missing_count`) ช่วยจับกรณีที่ระบบ Telemetry ขาดหาย

In [ ]:
def engineer_task1_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df[RAW_FEATURES].copy()
    eps = 1e-5
    
    # Missingness statistics per player
    out["missing_count"] = out[RAW_FEATURES].isna().sum(axis=1)
    
    # 1. Aimbot & Mechanical Discrepancy Ratios
    out["hs_to_accuracy_ratio"] = out["headshot_percentage"] / (out["accuracy_score"] + eps)
    out["mechanical_efficiency"] = (out["accuracy_score"] * out["headshot_percentage"]) / (out["reaction_time_ms"] + 1.0)
    out["kd_per_reaction_sec"] = out["kill_death_ratio"] / ((out["reaction_time_ms"] / 1000.0) + eps)
    out["aim_vs_gamesense_gap"] = out["accuracy_score"] - out["game_sense_score"]
    out["spray_vs_smoothness_ratio"] = out["spray_control_score"] / (out["aiming_smoothness"] + eps)
    
    # 2. Wallhack / Pre-Aim Suspicion Ratios
    out["preaim_to_map_ratio"] = out["crosshair_placement"] / (out["map_knowledge"] + eps)
    out["firstblood_to_survival_ratio"] = out["first_blood_rate"] / (out["survival_time_avg"] + eps)
    out["clutch_to_teamplay_ratio"] = out["clutch_success_rate"] / (out["team_play_score"] + eps)
    
    # 3. Account Maturity & Report Velocity
    out["reports_per_day"] = out["reports_received"] / (out["account_age_days"] + 1.0)
    out["reports_per_friend"] = out["reports_received"] / (out["friend_network_size"] + 1.0)
    out["device_changes_per_day"] = out["device_changes_count"] / (out["account_age_days"] + 1.0)
    out["daily_play_minutes"] = out["avg_session_length_min"] * out["sessions_per_day"]
    out["damage_to_kd_ratio"] = out["damage_per_round"] / (out["kill_death_ratio"] + eps)
    
    return out

X_train_fe = engineer_task1_features(train_df)
X_test_fe = engineer_task1_features(test_df)
y_train = train_df[TARGET_COL].values
FEATURE_COLS = list(X_train_fe.columns)

print(f"Engineered Feature Matrix — Train: {X_train_fe.shape}, Test: {X_test_fe.shape}")

## 5. Out-of-Fold Cross-Validation & Multi-Model Ensemble (LGBM + XGB + CatBoost)
เราใช้ **5-Fold Stratified Cross-Validation** (`SEED = 42`) เพื่อเปรียบเทียบโมเดลตั้งแต่ Ground Baseline (Logistic Regression, Random Forest) ไปจนถึง Gradient Boosted Decision Trees (LightGBM, XGBoost, CatBoost) และทำ **Probability Blending + Exact $F_2$ Threshold Optimization**

In [ ]:
def compute_f2_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    f2_bin = fbeta_score(y_true, y_pred, beta=2.0, pos_label=1, zero_division=0)
    f2_macro = fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0)
    return {
        "threshold": threshold,
        "f2_binary": f2_bin,
        "f2_macro": f2_macro,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }

def optimize_f2_threshold(y_true, y_prob, mode="binary"):
    thresholds = np.linspace(0.05, 0.90, 171)
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        score = (
            fbeta_score(y_true, y_pred, beta=2.0, pos_label=1, zero_division=0)
            if mode == "binary"
            else fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0)
        )
        if score > best_score:
            best_score = score
            best_t = float(t)
    return best_t, float(best_score)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_preds = {
    "logreg_baseline": np.zeros(len(train_df)),
    "lgbm": np.zeros(len(train_df)),
    "xgb": np.zeros(len(train_df)),
}
test_preds = {
    "logreg_baseline": np.zeros(len(test_df)),
    "lgbm": np.zeros(len(test_df)),
    "xgb": np.zeros(len(test_df)),
}

has_catboost = False
try:
    from catboost import CatBoostClassifier
    has_catboost = True
    oof_preds["catboost"] = np.zeros(len(train_df))
    test_preds["catboost"] = np.zeros(len(test_df))
except ImportError:
    pass

feature_importances = np.zeros(len(FEATURE_COLS))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train_fe, y_train), 1):
    X_tr, y_tr = X_train_fe.iloc[trn_idx], y_train[trn_idx]
    X_va, y_va = X_train_fe.iloc[val_idx], y_train[val_idx]
    
    # 1. Ground Baseline: Median Imputer + StandardScaler + Logistic Regression
    imp = SimpleImputer(strategy="median")
    scl = StandardScaler()
    X_tr_imp = scl.fit_transform(imp.fit_transform(X_tr))
    X_va_imp = scl.transform(imp.transform(X_va))
    X_te_imp = scl.transform(imp.transform(X_test_fe))
    
    lr = LogisticRegression(max_iter=500, class_weight="balanced", random_state=SEED)
    lr.fit(X_tr_imp, y_tr)
    oof_preds["logreg_baseline"][val_idx] = lr.predict_proba(X_va_imp)[:, 1]
    test_preds["logreg_baseline"] += lr.predict_proba(X_te_imp)[:, 1] / skf.n_splits
    
    # 2. LightGBM Model
    model_lgb = lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=63,
        subsample_for_bin=50000,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=40,
        reg_alpha=1.0,
        reg_lambda=2.0,
        random_state=SEED + fold,
        n_jobs=-1,
        verbose=-1
    )
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=False)]
    )
    oof_preds["lgbm"][val_idx] = model_lgb.predict_proba(X_va)[:, 1]
    test_preds["lgbm"] += model_lgb.predict_proba(X_test_fe)[:, 1] / skf.n_splits
    feature_importances += model_lgb.feature_importances_ / skf.n_splits
    
    # 3. XGBoost Model
    model_xgb = xgb.XGBClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=2.0,
        eval_metric="logloss",
        early_stopping_rounds=80,
        random_state=SEED + fold,
        n_jobs=-1,
        tree_method="hist"
    )
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_preds["xgb"][val_idx] = model_xgb.predict_proba(X_va)[:, 1]
    test_preds["xgb"] += model_xgb.predict_proba(X_test_fe)[:, 1] / skf.n_splits
    
    # 4. Optional CatBoost Model
    if has_catboost:
        model_cb = CatBoostClassifier(
            iterations=1500,
            learning_rate=0.04,
            depth=6,
            random_seed=SEED + fold,
            verbose=0
        )
        model_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=80, verbose=False)
        oof_preds["catboost"][val_idx] = model_cb.predict_proba(X_va)[:, 1]
        test_preds["catboost"] += model_cb.predict_proba(X_test_fe)[:, 1] / skf.n_splits
        
    print(f"Fold {fold}/5 completed | LGBM Val AUC: {roc_auc_score(y_va, oof_preds['lgbm'][val_idx]):.4f} | XGB Val AUC: {roc_auc_score(y_va, oof_preds['xgb'][val_idx]):.4f}")

# Ensemble Blend (Weighted probability average of GBDT models)
if has_catboost:
    oof_preds["ensemble"] = 0.45 * oof_preds["lgbm"] + 0.35 * oof_preds["xgb"] + 0.20 * oof_preds["catboost"]
    test_preds["ensemble"] = 0.45 * test_preds["lgbm"] + 0.35 * test_preds["xgb"] + 0.20 * test_preds["catboost"]
else:
    oof_preds["ensemble"] = 0.55 * oof_preds["lgbm"] + 0.45 * oof_preds["xgb"]
    test_preds["ensemble"] = 0.55 * test_preds["lgbm"] + 0.45 * test_preds["xgb"]

# Rule 2 Allowed Enhancement: Combine Full Train + Validation sets for final refit after CV validation
full_lgb = lgb.LGBMClassifier(
    n_estimators=700, learning_rate=0.03, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=40,
    reg_alpha=1.0, reg_lambda=2.0, random_state=SEED, n_jobs=-1, verbose=-1
)
full_lgb.fit(X_train_fe, y_train)
test_preds["ensemble"] = 0.80 * test_preds["ensemble"] + 0.20 * full_lgb.predict_proba(X_test_fe)[:, 1]

# Rule 3 Allowed Hyperparameter Optimization Helper (Optuna)
def run_optuna_lgbm_study(n_trials=20):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        params = {
            "n_estimators": 800,
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.08, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 127),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
            "subsample": trial.suggest_float("subsample", 0.6, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.95),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
            "random_state": SEED,
            "n_jobs": -1,
            "verbose": -1,
        }
        trn_idx, val_idx = next(skf.split(X_train_fe, y_train))
        clf = lgb.LGBMClassifier(**params)
        clf.fit(
            X_train_fe.iloc[trn_idx], y_train[trn_idx],
            eval_set=[(X_train_fe.iloc[val_idx], y_train[val_idx])],
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )
        probs = clf.predict_proba(X_train_fe.iloc[val_idx])[:, 1]
        _, best_f2 = optimize_f2_threshold(y_train[val_idx], probs, mode="binary")
        return best_f2
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    print("Best Optuna F2:", study.best_value, "| Params:", study.best_params)
    return study

## 6. Metric-Driven Decision Threshold Optimization ($F_2$ Score) & Diagnostics
เนื่องจากโจทย์ใช้ **$F_2$ Score** ซึ่งให้น้ำหนัก **Recall สูงกว่า Precision ถึง 2 เท่า** การใช้ Threshold มาตรฐานที่ `0.50` จะทำให้โมเดลพลาดคนโกง (False Negative) มากเกินไป เราจึงค้นหาจุดตัด $\tau^*$ บน Out-of-Fold Predictions ทั้งสำหรับ **Binary $F_2$** และ **Macro $F_2$** พร้อมบันทึกกราฟและตารางเปรียบเทียบลงใน `task1/assets/`

In [ ]:
benchmark_rows = []
for name, oof_prob in oof_preds.items():
    # Default 0.50 threshold
    m_def = compute_f2_metrics(y_train, oof_prob, threshold=0.50)
    m_def["model"] = f"{name} (t=0.50)"
    benchmark_rows.append(m_def)
    
    # Optimal Binary F2 threshold
    best_t_bin, _ = optimize_f2_threshold(y_train, oof_prob, mode="binary")
    m_opt_bin = compute_f2_metrics(y_train, oof_prob, threshold=best_t_bin)
    m_opt_bin["model"] = f"{name} (opt F2_bin t={best_t_bin:.3f})"
    benchmark_rows.append(m_opt_bin)
    
    # Optimal Macro F2 threshold
    best_t_mac, _ = optimize_f2_threshold(y_train, oof_prob, mode="macro")
    m_opt_mac = compute_f2_metrics(y_train, oof_prob, threshold=best_t_mac)
    m_opt_mac["model"] = f"{name} (opt F2_macro t={best_t_mac:.3f})"
    benchmark_rows.append(m_opt_mac)

benchmark_df = pd.DataFrame(benchmark_rows)[
    ["model", "threshold", "f2_binary", "f2_macro", "precision", "recall", "f1", "roc_auc", "pr_auc"]
]
benchmark_df.to_csv(ASSETS_DIR / "model-benchmark.csv", index=False)
display(benchmark_df.round(4))

# Plot Threshold Optimization Curve & Feature Importance
best_t_bin, best_f2_bin = optimize_f2_threshold(y_train, oof_preds["ensemble"], mode="binary")
best_t_mac, best_f2_mac = optimize_f2_threshold(y_train, oof_preds["ensemble"], mode="macro")

thresholds = np.linspace(0.05, 0.90, 171)
f2_bin_curve = [fbeta_score(y_train, (oof_preds["ensemble"] >= t).astype(int), beta=2.0, pos_label=1, zero_division=0) for t in thresholds]
f2_mac_curve = [fbeta_score(y_train, (oof_preds["ensemble"] >= t).astype(int), beta=2.0, average="macro", zero_division=0) for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].plot(thresholds, f2_bin_curve, label=f"Binary F2 (Peak={best_f2_bin:.4f} @ t={best_t_bin:.3f})", color="#DC2626", lw=2.5)
axes[0].plot(thresholds, f2_mac_curve, label=f"Macro F2 (Peak={best_f2_mac:.4f} @ t={best_t_mac:.3f})", color="#2563EB", lw=2.5)
axes[0].axvline(best_t_bin, color="#DC2626", linestyle="--", alpha=0.7)
axes[0].axvline(best_t_mac, color="#2563EB", linestyle="--", alpha=0.7)
axes[0].axvline(0.50, color="#6B7280", linestyle=":", label="Default Threshold (0.50)")
axes[0].set_title("Task 1: Out-of-Fold F2 Score vs Decision Threshold", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Decision Threshold (Probability Cut-off)")
axes[0].set_ylabel("F2 Score")
axes[0].legend()
axes[0].grid(alpha=0.3)

fi_df = pd.DataFrame({"feature": FEATURE_COLS, "importance": feature_importances}).sort_values("importance", ascending=False).head(15)
axes[1].barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color="#059669")
axes[1].set_title("Task 1: Top 15 Most Informative Anti-Cheat Features", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Average Split Importance (LightGBM 5-Fold)")
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(ASSETS_DIR / "threshold-tuning-and-feature-importance.png", dpi=300, bbox_inches="tight")
plt.show()

# Confusion Matrix at Optimal F2 Threshold
fig, ax = plt.subplots(figsize=(5.5, 4.8))
cm = confusion_matrix(y_train, (oof_preds["ensemble"] >= best_t_bin).astype(int))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Pred Legit (0)", "Pred Cheater (1)"],
            yticklabels=["Actual Legit (0)", "Actual Cheater (1)"], ax=ax)
ax.set_title(f"Task 1 OOF Confusion Matrix (Threshold = {best_t_bin:.3f})", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(ASSETS_DIR / "confusion-matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Save Out-of-Fold & Final Test Predictions (`task1/predictions/`)
บันทึกผลทำนายความน่าจะเป็นและคลาสสุดท้ายลงใน `task1/predictions/test_predictions.csv` (จำนวน `25,889` แถวครบทุก `player_id`) พร้อมนำไปรวมใน `sample_submission.csv`

In [ ]:
# Save OOF predictions
oof_out = pd.DataFrame({
    "player_id": train_df[ID_COL],
    "is_cheater_true": y_train,
    "prob_ensemble": oof_preds["ensemble"],
    "pred_f2_binary": (oof_preds["ensemble"] >= best_t_bin).astype(int),
    "pred_f2_macro": (oof_preds["ensemble"] >= best_t_mac).astype(int),
})
oof_out.to_csv(PREDICTIONS_DIR / "oof_predictions.csv", index=False)

# Save Test predictions (defaulting to optimal F2 binary threshold, and also storing macro variant)
test_out = pd.DataFrame({
    "player_id": test_df[ID_COL],
    "prob_cheater": test_preds["ensemble"],
    "task1": (test_preds["ensemble"] >= best_t_bin).astype(int),
    "task1_macro_opt": (test_preds["ensemble"] >= best_t_mac).astype(int),
})
test_out.to_csv(PREDICTIONS_DIR / "test_predictions.csv", index=False)

print(f"Saved OOF predictions  : {PREDICTIONS_DIR / 'oof_predictions.csv'} ({len(oof_out):,} rows)")
print(f"Saved Test predictions : {PREDICTIONS_DIR / 'test_predictions.csv'} ({len(test_out):,} rows)")
print("Predicted Test Cheater Ratio (F2 Binary Opt):", test_out["task1"].mean().round(4))
print("Predicted Test Cheater Ratio (F2 Macro Opt) :", test_out["task1_macro_opt"].mean().round(4))